In [1]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
import optuna

from catboost import Pool, CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from optuna.pruners import MedianPruner
from optuna.integration import CatBoostPruningCallback

try:
    import pyarrow
    print("PyArrow is installed. Version:", pyarrow.__version__)
except ModuleNotFoundError:
    print("PyArrow is NOT found in this environment. You installed it in a different one.")

PyArrow is installed. Version: 23.0.1


In [2]:
X = pd.read_parquet('../modified_data/train_no_nan_min_var.parquet')
y = pd.read_parquet('../data/train_target.parquet')


test_main = pd.read_parquet('../data/test_main_features.parquet')
test_extra = pd.read_parquet('../data/test_extra_features.parquet')



print('Тренировочные данные:', X.shape)
print('Тестовые данные:', test_main.shape)
print('Тестовые данные:', test_extra.shape)

Тренировочные данные: (750000, 2271)
Тестовые данные: (250000, 200)
Тестовые данные: (250000, 2242)


In [3]:
cat_feature_names = [
    col_name for col_name in X.columns 
    if col_name.startswith("cat_feature")
]

In [4]:
# train = pd.merge(train_main, train_extra, on="customer_id", how="left")
test  = pd.merge(test_main,  test_extra,  on="customer_id", how="left")

In [5]:
test[X.columns]

,customer_id,cat_feature_1,cat_feature_2,cat_feature_3,cat_feature_4,cat_feature_5,cat_feature_6,cat_feature_7,cat_feature_8,cat_feature_9,...,num_feature_2359,num_feature_2362,num_feature_2363,num_feature_2364,num_feature_2365,num_feature_2368,num_feature_2369,num_feature_2370,num_feature_2371,num_feature_2372
0,1750001,1.0,0.0,2.0,0.0,2.0,3.0,2.0,2.0,4.0,...,-0.023246,-0.020769,-0.14374,NaN,-0.00276,-0.030607,-0.041319,-0.404816,-0.395589,NaN
1,1750002,0.0,0.0,2.0,0.0,2.0,3.0,2.0,2.0,4.0,...,-0.023246,-0.020769,-0.14374,-0.204497,-0.00276,-0.030607,-0.024319,-0.404816,0.048602,NaN
2,1750003,1.0,0.0,2.0,1.0,2.0,3.0,2.0,2.0,4.0,...,-0.023246,-0.020769,-0.14374,-0.204497,-0.00276,NaN,-0.044563,-0.404816,0.492792,NaN
3,1750004,1.0,0.0,2.0,0.0,2.0,3.0,2.0,2.0,4.0,...,-0.023246,-0.020769,-0.14374,NaN,-0.00276,-0.030607,-0.042305,-0.404816,NaN,NaN
4,1750005,0.0,1.0,2.0,0.0,2.0,3.0,2.0,2.0,4.0,...,-0.023246,-0.020769,-0.14374,NaN,-0.00276,-0.030607,-0.029387,-0.404816,-0.395589,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,1999996,1.0,2.0,2.0,1.0,2.0,3.0,2.0,2.0,4.0,...,NaN,NaN,NaN,NaN,-0.00276,NaN,-0.058314,-0.404816,NaN,NaN
249996,1999997,1.0,2.0,0.0,1.0,0.0,3.0,0.0,0.0,4.0,...,NaN,NaN,NaN,NaN,-0.00276,-0.030607,-0.010714,-0.404816,0.936983,0.044761
249997,1999998,1.0,2.0,2.0,1.0,2.0,3.0,2.0,2.0,4.0,...,NaN,NaN,NaN,NaN,-0.00276,NaN,-0.058314,-0.404816,NaN,NaN
249998,1999999,1.0,0.0,2.0,1.0,2.0,3.0,2.0,2.0,0.0,...,-0.023246,-0.020769,-0.14374,NaN,-0.00276,NaN,-0.056636,-0.404816,-0.395589,NaN


In [6]:
X[cat_feature_names] = X[cat_feature_names].astype(str)
test[cat_feature_names] = test[cat_feature_names].astype(str)

In [7]:
for c in cat_feature_names:
    X[c] = X[c].fillna("__MISSING__").astype(str)
    test[c]  = test[c].fillna("__MISSING__").astype(str)

In [8]:
for c in cat_feature_names:
    freq = X[c].value_counts(dropna=False)
    X[c + "__freq"] = X[c].map(freq).fillna(0).astype("int32")
    test[c + "__freq"]  = test[c].map(freq).fillna(0).astype("int32")

In [9]:
train, val, target, val_target = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [10]:
train_pool = Pool(data = train.drop("customer_id", axis=1), 
                  label = target.drop("customer_id", axis=1), 
                  cat_features = cat_feature_names)

In [11]:
val_pool = Pool(data = val.drop("customer_id", axis=1), 
                  label = val_target.drop("customer_id", axis=1), 
                  cat_features = cat_feature_names)

In [12]:
def objective(trial: optuna.Trial):
    params = {
        "loss_function":"MultiLogloss",
        
        "iterations":2000,         
        "learning_rate":trial.suggest_float( "lr", 3e-6, 0.1, log=True),       # стабильнее, чем 0.1
        "depth":trial.suggest_int("depth", 4, 12),                  # лучше для 2200 фичей
        
        "l2_leaf_reg": trial.suggest_float("l2", 4, 50.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.0, 5.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 2.0),
        "od_type": "Iter",
        "od_wait": 200,
        "use_best_model": True,
    
        
        "random_seed":42,
        "task_type":"GPU",
        "allow_writing_files":False,
        "verbose":200,
        "devices":'0',
    }


    model = CatBoostClassifier(**params)

    model.fit(train_pool, eval_set=val_pool)

    return model.get_best_score()["validation"]["MultiLogloss"]


In [ ]:

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

[I 2026-03-02 00:18:46,812] A new study created in memory with name: no-name-66df2ca6-4403-4b0a-8e8c-fbae160d7bb9


0:	learn: 0.6931363	test: 0.6931310	best: 0.6931310 (0)	total: 372ms	remaining: 12m 23s
200:	learn: 0.6900540	test: 0.6900531	best: 0.6900531 (200)	total: 1m 13s	remaining: 10m 59s
400:	learn: 0.6869890	test: 0.6869893	best: 0.6869893 (400)	total: 2m 27s	remaining: 9m 46s
600:	learn: 0.6839406	test: 0.6839406	best: 0.6839406 (600)	total: 3m 40s	remaining: 8m 33s
800:	learn: 0.6809074	test: 0.6809075	best: 0.6809075 (800)	total: 4m 54s	remaining: 7m 20s
1000:	learn: 0.6778890	test: 0.6778892	best: 0.6778892 (1000)	total: 6m 7s	remaining: 6m 6s
1200:	learn: 0.6748853	test: 0.6748863	best: 0.6748863 (1200)	total: 7m 21s	remaining: 4m 53s
1400:	learn: 0.6718983	test: 0.6718986	best: 0.6718986 (1400)	total: 8m 34s	remaining: 3m 39s
1600:	learn: 0.6689246	test: 0.6689251	best: 0.6689251 (1600)	total: 9m 47s	remaining: 2m 26s
1800:	learn: 0.6659662	test: 0.6659666	best: 0.6659666 (1800)	total: 11m 1s	remaining: 1m 13s


[I 2026-03-02 00:31:06,498] Trial 0 finished with value: 0.6630381770833333 and parameters: {'lr': 6.466440873073841e-06, 'depth': 5, 'l2': 38.090662067839546, 'random_strength': 1.3327737428979876, 'bagging_temperature': 0.7781777305829931}. Best is trial 0 with value: 0.6630381770833333.


1999:	learn: 0.6630376	test: 0.6630382	best: 0.6630382 (1999)	total: 12m 14s	remaining: 0us
bestTest = 0.6630381771
bestIteration = 1999
0:	learn: 0.6929726	test: 0.6929777	best: 0.6929777 (0)	total: 350ms	remaining: 11m 39s
200:	learn: 0.6599214	test: 0.6599188	best: 0.6599188 (200)	total: 1m 6s	remaining: 9m 51s
400:	learn: 0.6286764	test: 0.6286727	best: 0.6286727 (400)	total: 2m 11s	remaining: 8m 45s
600:	learn: 0.5991799	test: 0.5991762	best: 0.5991762 (600)	total: 3m 17s	remaining: 7m 39s
800:	learn: 0.5713526	test: 0.5713485	best: 0.5713485 (800)	total: 4m 23s	remaining: 6m 33s
1000:	learn: 0.5451173	test: 0.5451130	best: 0.5451130 (1000)	total: 5m 28s	remaining: 5m 28s
1200:	learn: 0.5203911	test: 0.5203865	best: 0.5203865 (1200)	total: 6m 34s	remaining: 4m 22s
1400:	learn: 0.4971076	test: 0.4971026	best: 0.4971026 (1400)	total: 7m 40s	remaining: 3m 16s
1600:	learn: 0.4751931	test: 0.4751876	best: 0.4751876 (1600)	total: 8m 46s	remaining: 2m 11s
1800:	learn: 0.4545747	test: 0.4

[I 2026-03-02 00:42:09,957] Trial 1 finished with value: 0.4352671354166667 and parameters: {'lr': 7.049649288910713e-05, 'depth': 4, 'l2': 4.652517524533346, 'random_strength': 1.9250778719833028, 'bagging_temperature': 0.47034539117260366}. Best is trial 1 with value: 0.4352671354166667.


1999:	learn: 0.4352746	test: 0.4352671	best: 0.4352671 (1999)	total: 10m 58s	remaining: 0us
bestTest = 0.4352671354
bestIteration = 1999


In [ ]:
study.best_params

In [ ]:
test_pool = Pool(data = test.drop("customer_id", axis = 1), 
                 cat_features = cat_feature_names)

In [ ]:
test_predict = model.predict(test_pool, prediction_type = "RawFormulaVal")

test_predict.shape

In [ ]:
predict_schema = [col.replace("target_", "predict_") for col in target.columns if col.startswith("target_")]

catboost_predictions = pl.DataFrame(test_predict, schema = predict_schema)

catboost_predictions.head(n = 5)

проверим roc_auc на тренировочной выборке

In [ ]:
y_true = target.drop('customer_id', axis = 1)
y_pred = model.predict(train_pool, prediction_type = 'RawFormulaVal')
# roc_auc_score(y_true, y_pred, average="macro")

roc_auc_score(y_true, y_pred, average="macro")

сохраняем сабмит для сдачи

In [ ]:
sample_submit = pd.read_parquet('./submits/sample_submit.parquet')


In [ ]:
timestamp = time.time()

In [ ]:
result_df = sample_submit.copy()
result_df.iloc[:, 1:] = test_predict
result_df['customer_id'] = result_df['customer_id'].astype('int32')
current_type = result_df['customer_id'].dtype

result_df.to_parquet(f'./submits/exp_{timestamp}.parquet', index=False)

сохраняем модельку))))))

In [ ]:
model.save_model(f"./models/exp_{timestamp}.cbm")